In [34]:
# Training data
training_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 3, 'Apple'],
    ['Red', 1, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],
]



In [35]:
# Column labels
header = ["color", "diameter", "label"]

In [36]:
# Count class labels
def class_counts(rows):
    counts = {}
    for row in rows:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts

In [37]:
# Question class
class Question:
    def __init__(self, column, value):
        self.column = column
        self.value = value

    def match(self, example):
        val = example[self.column]
        if isinstance(val, int):
            return val >= self.value
        else:
            return val == self.value

    def __repr__(self):
        condition = "=="
        if isinstance(self.value, int):
            condition = ">="
        return f"Is {header[self.column]} {condition} {str(self.value)}?"

In [38]:
# Partition dataset
def partition(rows, question):
    true_rows, false_rows = [], []
    for row in rows:
        if question.match(row):
            true_rows.append(row)
        else:
            false_rows.append(row)
    return true_rows, false_rows

In [39]:
# Leaf node
class Leaf:
    def __init__(self, rows):
        self.predictions = class_counts(rows)

In [40]:

# Decision node
class DecisionNode:
    def __init__(self, question, true_branch, false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

In [41]:

# Build simple tree manually
q1 = Question(1, 3)           # Is diameter >= 3 ?
q2 = Question(0, 'Yellow')   # Is color == Yellow ?

In [42]:
# Leaves
leaf1 = Leaf([['Yellow',3,'Mango'], ['Yellow',3,'Lemon']])
leaf2 = Leaf([['Green',3,'Mango']])
leaf3 = Leaf([['Red',1,'Grape'], ['Red',1,'Grape']])

In [43]:
# Tree structure
tree = DecisionNode(
    q1,
    DecisionNode(q2, leaf1, leaf2),
    leaf3
)

In [44]:
# Print tree
def print_tree(node, spacing=""):
    if isinstance(node, Leaf):
        print(spacing + "Predict", node.predictions)
        return

    print(spacing + str(node.question))

    print(spacing + '--> True:')
    print_tree(node.true_branch, spacing + "  ")

    print(spacing + '--> False:')
    print_tree(node.false_branch, spacing + "  ")

In [45]:
print_tree(tree)

Is diameter >= 3?
--> True:
  Is color == Yellow?
  --> True:
    Predict {'Mango': 1, 'Lemon': 1}
  --> False:
    Predict {'Mango': 1}
--> False:
  Predict {'Grape': 2}


             diameter ≥ 3 ?
              /       \
           Yes         No
            |           |
      color == Yellow?   Grape
        /      \
     Yes        No
      |           |
      Mango/Lemon    Mango



In [46]:
# Classify function
def classify(row, node):

    if isinstance(node, Leaf):
        return node.predictions

    if node.question.match(row):
        return classify(row, node.true_branch)
    else:
        return classify(row, node.false_branch)


In [47]:
# Convert prediction to percentage
def print_leaf(counts):
    total = sum(counts.values()) * 1.0
    probs = {}

    for lbl in counts.keys():
        probs[lbl] = str(int(counts[lbl] / total * 100)) + "%"

    return probs

In [48]:
# Actual vs Predicted
for row in training_data:
    print("Actual:", row[-1], 
          "Predicted:", print_leaf(classify(row, tree)))

Actual: Apple Predicted: {'Mango': '100%'}
Actual: Apple Predicted: {'Mango': '50%', 'Lemon': '50%'}
Actual: Grape Predicted: {'Grape': '100%'}
Actual: Grape Predicted: {'Grape': '100%'}
Actual: Lemon Predicted: {'Mango': '50%', 'Lemon': '50%'}
